In [1]:
reviews = "../results/training_review"

In [2]:
import pandas as pd
df = pd.read_csv("../results/bench_scores_deepreview_flash.csv")

In [3]:
def get_gt(review_filename):
    paper_id = review_filename.split(".")[0]
    row = df[df["paper_id"] == paper_id]
    return row["gt_avg_score"]

In [ ]:
rl_prompt = """
You will get a review of a paper and a set of retrieved anchor reviews, based on the anchor review, estimate the score of the paper under review. The score should be between 1 and 10, where 1 is the worst and 10 is the best. Round to the nearest .5 or .0. 

Scoring rules:
- Your final score must be positioned relative to the retrieved anchors.
- Do not pick a score first and then justify it. Compare to anchors first, let the comparison set the score.
- The number of weaknesses listed is not a signal for a bad paper — focus on weakness content and anchor scores.
- Score distribution: extreme scores are rare but valid. If the paper truly is exceptional or truly weak, give an extreme score even if most retrieved anchors sit in the middle.
- Do NOT cluster scores around 5, the score should be relative to the retrieval samples. Score a good paper high and a bad paper low. 
- Compare the paper under review with every single anchor paper

Scoring scale:
1 - strong reject
3 - reject
4 - borderline reject
6 - borderline accept
8 - accept
10 - strong accept

Give your analysis first, then put your final score in a XML-style tag <score></score>
"""

In [5]:
from openai import OpenAI
import os

or_client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv("OPENROUTER_API_KEY")
)

In [6]:
import pickle
emb_path = "../datasets/human_reviews_embeddings_deepreview.pkl"
idx_path = "../datasets/human_review_score_index_deepreview.pkl"

with open(emb_path, "rb") as f:
    review_embeddings = pickle.load(f)

with open(idx_path, "rb") as f:
    review_score_index = pickle.load(f)


In [8]:
import os
from concurrent.futures import ThreadPoolExecutor
import numpy as np
bin_names = ["very_low", "low", "medium", "high", "very_high"]
thresholds = [2, 4, 6, 8]

bins = {name: [] for name in bin_names}
bins_embeddings = {name: [] for name in bin_names}

for i in review_score_index:
    score = review_score_index[i]
    idx = next((j for j, t in enumerate(thresholds) if score <= t), len(thresholds))
    name = bin_names[idx]
    bins[name].append(i)
    bins_embeddings[name].append(review_embeddings[i])

for name in bin_names:
    bins_embeddings[name] = np.array(bins_embeddings[name])

def build_sample(review):
    with open(os.path.join(reviews, review), 'r') as f:
        review_content = f.read().split("Score and Decision")[0]

    query_embedding = or_client.embeddings.create(
        model="google/gemini-embedding-001",
        input=review_content,
        encoding_format="float",
    )
    query_vector = np.array(query_embedding.data[0].embedding)

    anchor_samples = []
    for name in bin_names:
        if len(bins_embeddings[name]) == 0:
            continue
        similarities = bins_embeddings[name] @ query_vector.T
        top_indices = np.argsort(similarities)[-2:]
        selected = [bins[name][idx] for idx in top_indices]

        for filename in selected:
            if filename == review:
                continue # skip itself
            with open(os.path.join("../datasets/deepreview_13k_train/human_reviews", filename), 'r') as f:
                anchor_samples.append(f.read().split("Score and Decision")[0])

    gt = get_gt(review)
    return {
        "prompt": [
                    {
                        "content": rl_prompt,
                        "role": "system"
                    },
                    {
                        "content": f"Paper review:\n{review_content}\n\nAnchor reviews:\n" + "\n\n".join(anchor_samples),
                        "role": "user"
                    }
                ], 
        "solution": float(gt.values[0]),
        "paper_id": review.split(".")[0]
    }
import tqdm
with ThreadPoolExecutor(max_workers=50) as executor:
    ds = list(tqdm.tqdm(executor.map(build_sample, os.listdir(reviews)), total=len(os.listdir(reviews))))

100%|██████████| 1894/1894 [01:02<00:00, 30.26it/s]


In [ ]:
from datasets import Dataset
ds_hf = Dataset.from_list(ds)
ds_hf.push_to_hub("weathon/grpo_dataset")

In [9]:
import json
with open("grpo_dataset.json", "w") as f:
    json.dump(ds, f, indent=4)

In [92]:
def rollout(messages):
    _response = or_client.chat.completions.create(
        model="deepseek/deepseek-v4-flash",
        messages=messages,
        extra_body={"reasoning": {"enabled": False, "effort": "low"}, "provider": {"only": ["deepseek"]}}
    )
    response = _response.choices[0].message
    return response

In [93]:
idx = 765 
messages = ds[idx]["prompt"]

messages[0]["content"] = messages[0]["content"]#.replace("Give your analysis first, then put your final score in a XML-style tag <score></score>", "Put your final score in a XML-style tag <score></score>")

with ThreadPoolExecutor(max_workers=50) as executor:
    rollouts = list(executor.map(rollout, [messages] * 5))

In [94]:
ds[idx]["solution"]

3.0

In [95]:
rollouts

[ChatCompletionMessage(content="Looking at the anchor reviews, I need to compare the paper under review against each one to determine where it falls on the score distribution.\n\n**Anchor 1 - MixAttention (Score range: 1-3):** This paper was rejected for lack of novelty and being a straightforward combination of existing techniques. The current paper has more novelty (multi-task decomposition of audio instruction generation, cloud-edge architecture) and a large-scale real-world deployment (600M segments). The current paper is clearly stronger.\n\n**Anchor 2 - Long Horizon Episodic Decision Making (Score range: 1-3):** This paper was very poorly written, lacked clarity, and had major methodological issues. The current paper is well-written, has clear methodology, and substantial real-world validation. Much stronger.\n\n**Anchor 3 - VLA4CD (Score range: 3-5):** This paper had issues with motivation and experimental validation. The current paper has a clearer motivation and much more conv

In [85]:
print(response.content) 

Looking at the anchor papers for score calibration:

- **Cut Your Losses** (scores: 10, 6, 8, 10) — High scores, accepted at top venue. Novel algorithm with massive practical impact (memory reduction from 24GB to 1MB). Strong contribution.
- **TANGO** (scores: 10, 8, 8, 8) — Accepted. Strong technical contribution, comprehensive experiments, open-source.
- **RouteFormer** (scores: 6, 8, 6) — Accepted. Solid contribution with novel dataset and metric.
- **MANGO** (scores: 6, 5, 6, 8) — Rejected despite some high scores. Benchmark paper with limited novelty.
- **GPT-Driver** (scores: 5, 5, 5, 5) — Rejected. Simple application of existing API, limited novelty.
- **MixAttention** (scores: 1, 3, 3, 1) — Strong reject. No novelty, combination of existing techniques without insight.
- **Long Horizon Episodic Decision Making** (scores: 1, 1, 1, 3) — Strong reject. Poorly written, early stage.
- **VLA4CD** (scores: 3, 3, 5) — Reject. Weak motivation, unclear contribution.

Now comparing the pap